# ETL Pipeline for Data Preprocessing, Transformation, and Loading

This notebook demonstrates an automated ETL (Extract, Transform, Load) pipeline using Pandas and Scikit-learn.

## 1. Import Required Libraries

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
import os

## 2. Load Raw Data

In [ ]:
def generate_sample_data(file_path):
    """Generates sample data if input file doesn't exist."""
    data = {
        'age': [25, 30, np.nan, 45, 50],
        'salary': [50000, 60000, 70000, np.nan, 90000],
        'department': ['HR', 'IT', 'Finance', 'IT', 'HR'],
        'experience': [2, 5, 8, 10, np.nan]
    }
    df = pd.DataFrame(data)
    df.to_csv(file_path, index=False)
    print(f"Sample data generated and saved to: {file_path}")
    return df

def extract_data(file_path):
    """Extracts data from a CSV file."""
    if not os.path.exists(file_path):
        print(f"Input file '{file_path}' not found. Generating sample data...")
        return generate_sample_data(file_path)
    try:
        data = pd.read_csv(file_path)
        print("Data extraction successful.")
        return data
    except Exception as e:
        print(f"Error during extraction: {e}")
        return None

# Define file paths
INPUT_FILE = 'dataset.csv'

# Extract data
raw_data = extract_data(INPUT_FILE)
raw_data.head()

## 3. Data Preprocessing

In [ ]:
# Basic preprocessing: check for duplicates, data types
print("Data info before preprocessing:")
print(raw_data.info())
print("\nMissing values:")
print(raw_data.isnull().sum())

# Remove duplicates if any
raw_data = raw_data.drop_duplicates()
print(f"\nData shape after removing duplicates: {raw_data.shape}")

# For this example, we'll handle missing values in the transformation step

## 4. Data Transformation

In [ ]:
# Identify numerical and categorical columns
num_cols = raw_data.select_dtypes(include=['float64', 'int64']).columns.tolist()
cat_cols = raw_data.select_dtypes(include=['object']).columns.tolist()

print(f"Numerical columns: {num_cols}")
print(f"Categorical columns: {cat_cols}")

# Manual transformation (will be automated in pipeline)
# Handle missing values
imputer_num = SimpleImputer(strategy='mean')
raw_data[num_cols] = imputer_num.fit_transform(raw_data[num_cols])

# Encode categorical
le = LabelEncoder()
for col in cat_cols:
    raw_data[col] = le.fit_transform(raw_data[col].astype(str))

# Scale numerical
scaler = StandardScaler()
raw_data[num_cols] = scaler.fit_transform(raw_data[num_cols])

print("Manual transformation complete.")
raw_data.head()

## 5. Build ETL Pipeline with Scikit-learn

In [ ]:
# Reload raw data for pipeline
raw_data = extract_data(INPUT_FILE)

# Define transformers
numerical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='mean')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore'))
])

# Combine transformers
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numerical_transformer, num_cols),
        ('cat', categorical_transformer, cat_cols)
    ])

# Full pipeline
etl_pipeline = Pipeline(steps=[('preprocessor', preprocessor)])

# Fit and transform
transformed_data = etl_pipeline.fit_transform(raw_data)

print("Pipeline transformation complete.")
print(f"Transformed data shape: {transformed_data.shape}")

# Convert back to DataFrame for saving
# Get feature names
num_features = num_cols
cat_features = etl_pipeline.named_steps['preprocessor'].named_transformers_['cat'].named_steps['encoder'].get_feature_names_out(cat_cols)
all_features = num_features + cat_features.tolist()

transformed_df = pd.DataFrame(transformed_data, columns=all_features)
transformed_df.head()

## 6. Automate ETL Process

In [ ]:
def run_etl_pipeline(input_file, output_file):
    """
    Automated ETL pipeline function.
    Extracts data, transforms it using scikit-learn pipeline, and loads to CSV.
    """
    # Extract
    data = extract_data(input_file)
    if data is None:
        return None
    
    # Identify columns
    num_cols = data.select_dtypes(include=['float64', 'int64']).columns.tolist()
    cat_cols = data.select_dtypes(include=['object']).columns.tolist()
    
    # Build pipeline
    numerical_transformer = Pipeline(steps=[
        ('imputer', SimpleImputer(strategy='mean')),
        ('scaler', StandardScaler())
    ])
    
    categorical_transformer = Pipeline(steps=[
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('encoder', OneHotEncoder(handle_unknown='ignore'))
    ])
    
    preprocessor = ColumnTransformer(
        transformers=[
            ('num', numerical_transformer, num_cols),
            ('cat', categorical_transformer, cat_cols)
        ])
    
    pipeline = Pipeline(steps=[('preprocessor', preprocessor)])
    
    # Transform
    transformed = pipeline.fit_transform(data)
    
    # Get feature names for DataFrame
    num_features = num_cols
    if cat_cols:
        cat_features = pipeline.named_steps['preprocessor'].named_transformers_['cat'].named_steps['encoder'].get_feature_names_out(cat_cols)
        all_features = num_features + cat_features.tolist()
    else:
        all_features = num_features
    
    transformed_df = pd.DataFrame(transformed, columns=all_features)
    
    # Load
    transformed_df.to_csv(output_file, index=False)
    print(f"ETL pipeline completed. Transformed data saved to: {output_file}")
    
    return transformed_df

# Run the automated pipeline
OUTPUT_FILE = 'transformed_data.csv'
final_data = run_etl_pipeline(INPUT_FILE, OUTPUT_FILE)
final_data.head()

## 7. Save Transformed Data

In [ ]:
# The transformed data has been saved in the automated function
# Let's verify the output file
if os.path.exists(OUTPUT_FILE):
    saved_data = pd.read_csv(OUTPUT_FILE)
    print(f"Output file exists with shape: {saved_data.shape}")
    print("First few rows:")
    print(saved_data.head())
else:
    print("Output file not found.")